# DiD-BCF — D_contamination (linearity_degree=3)

**Workstream D · staggered adoption (cohort x event-time effects)**

stronger dynamics -> larger TWFE contamination

Fits DiD-BCF on the `D_contamination` scenario at `linearity_degree=3` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [1]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 13.6 MB/s eta 0:00:00


In [2]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

Using DiD-BCF engine at: /content/DiD-BCF/Simulation_Studies_Revision


In [3]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_contamination",
    linearity_degree=3,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

[D_contamination_lin_3] staggered DGP | N=(200,) | reps=100 | 100 fits | jobs=1


D_contamination: 100%|██████████| 100/100 [6:17:29<00:00, 226.49s/fit]

[D_contamination_lin_3] wrote 3000 rows -> /content/DiD-BCF/Simulation_Studies_Revision/Results/summaries_D_contamination_lin_3.csv


,dgp,setting,linearity_degree,N,rep,estimand_type,estimand_id,g,t,k,...,p_bayes,surf_rmse,surf_mae,surf_n,surf_mape,surf_cover95,surf_len95,surf_cover90,surf_len90,true
0,staggered,D_contamination,3,200,0,GATT,g=4_t=4,4.0,4.0,0.0,...,0.283333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.732024
1,staggered,D_contamination,3,200,0,GATT,g=4_t=5,4.0,5.0,1.0,...,0.098667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.917643
2,staggered,D_contamination,3,200,0,GATT,g=4_t=6,4.0,6.0,2.0,...,0.067333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.103263
3,staggered,D_contamination,3,200,0,GATT,g=4_t=7,4.0,7.0,3.0,...,0.010667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.288882
4,staggered,D_contamination,3,200,0,GATT,g=5_t=5,5.0,5.0,0.0,...,0.038667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.529720


In [4]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

,dgp,setting,linearity_degree,N,estimand_type,estimand_id,role,bias_corrected,bias_plain,cover90_corrected,...,len95_corrected,len95_plain,mae_corrected,mae_plain,reject05_corrected,reject05_plain,rmse_corrected,rmse_plain,sd_ratio_corrected,sd_ratio_plain
0,staggered,D_contamination,3,200,ATT,ATT,power,-2.440961,-5.897233,0.00,...,0.931486,3.343956,2.440961,5.897233,1.0,0.93,2.519121,5.920147,0.346953,1.653976
1,staggered,D_contamination,3,200,ES,k=0,power,-0.943359,-4.531156,0.07,...,0.850784,1.798026,0.953471,4.531156,1.0,0.46,1.037020,4.542527,0.459495,1.870364
2,staggered,D_contamination,3,200,ES,k=1,power,-3.274624,-7.009358,0.00,...,1.067132,3.447413,3.274624,7.009358,1.0,0.95,3.348402,7.037018,0.361360,1.558862
3,staggered,D_contamination,3,200,ES,k=2,power,-3.459784,-6.743059,0.00,...,1.349083,4.801801,3.459784,6.743059,1.0,0.95,3.553945,6.783574,0.367991,1.559112
4,staggered,D_contamination,3,200,ES,k=3,power,-2.346656,-4.982701,0.03,...,1.771371,5.903510,2.346656,4.982701,1.0,0.93,2.503012,5.030774,0.461311,1.970374
5,staggered,D_contamination,3,200,GATT,g=4_t=4,power,-0.087651,-2.747851,0.82,...,1.434863,3.003739,0.362908,2.747851,1.0,0.00,0.449158,2.750570,0.796351,33.552732
6,staggered,D_contamination,3,200,GATT,g=4_t=5,power,-1.192661,-3.917279,0.17,...,1.478706,3.847651,1.196330,3.917279,1.0,0.04,1.334027,3.946173,0.583810,1.967090
7,staggered,D_contamination,3,200,GATT,g=4_t=6,power,-1.974920,-4.657556,0.08,...,1.586731,5.128353,1.974920,4.657556,1.0,0.40,2.146876,4.710149,0.449926,1.799293
8,staggered,D_contamination,3,200,GATT,g=4_t=7,power,-2.346656,-4.982701,0.03,...,1.771371,5.903510,2.346656,4.982701,1.0,0.93,2.503012,5.030774,0.461311,1.970374
9,staggered,D_contamination,3,200,GATT,g=5_t=5,power,-0.623416,-4.476586,0.55,...,1.697787,2.479241,0.660937,4.476586,1.0,0.42,0.789554,4.503960,0.806606,1.376300


## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [5]:
surface_metrics(summaries)

,dgp,setting,linearity_degree,N,method,n_reps,avg_n_treated_obs,surf_rmse_mean,surf_rmse_sd,surf_mae_mean,...,surf_mape_mean,surf_mape_sd,surf_cover90_mean,surf_cover90_sd,surf_cover95_mean,surf_cover95_sd,surf_len90_mean,surf_len90_sd,surf_len95_mean,surf_len95_sd
0,staggered,D_contamination,3,200,corrected,100,431.6,4.272182,0.470464,3.100193,...,0.337470,0.026367,0.186715,0.030828,0.222532,0.034547,1.490477,0.128053,1.798445,0.162726
1,staggered,D_contamination,3,200,plain,100,431.6,6.773138,0.565914,5.897235,...,0.726841,0.060909,0.042422,0.034414,0.079843,0.048265,3.940649,0.418352,4.733359,0.485611


## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [6]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=3)
bacon_summary(df0)

{'twfe': 5.063198924628939,
 'bacon_total': 5.063198924628939,
 'weight_treated_vs_untreated': 0.6334173693819004,
 'weight_earlier_vs_later': 0.2391665942217972,
 'weight_already_treated': 0.1274160363963024,
 'components':                    type  treat_group  control_group    weight        dd  \
 0      Earlier_vs_Later          4.0            5.0  0.060205  2.641511   
 1      Earlier_vs_Later          4.0            6.0  0.106778  3.036811   
 2      Earlier_vs_Later          5.0            6.0  0.072184  4.176324   
 3      Later_vs_Earlier          5.0            4.0  0.045153  3.585888   
 4      Later_vs_Earlier          6.0            4.0  0.053389  4.402213   
 5      Later_vs_Earlier          6.0            5.0  0.028874  4.134568   
 6  Treated_vs_Untreated          4.0            inf  0.231731  4.719428   
 7  Treated_vs_Untreated          5.0            inf  0.234982  6.264973   
 8  Treated_vs_Untreated          6.0            inf  0.166704  7.176302   
 
    contributi